# Statistical testing

Does income actually predict how much a customer spends? Does campaign
response really differ by education? Formal tests rather than eyeballed
bar charts.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

customers = pd.read_csv('outputs/customers_clean.csv', parse_dates = ['Dt_Customer'])
customers.shape

(2216, 31)

In [2]:
r_income_spend, p_income_spend = stats.pearsonr(customers['Income'], customers['TotalSpend'])
print(f'income vs total spend: r = {r_income_spend:.3f}, p = {p_income_spend:.3e}')

income vs total spend: r = 0.668, p = 5.844e-286


In [3]:
groups = [customers.loc[customers['Education'] == e, 'Response'] for e in customers['Education'].unique()]
f_stat, p_edu = stats.f_oneway(*groups)
print(f'ANOVA, Response across education levels: F = {f_stat:.2f}, p = {p_edu:.3e}')

for e in customers['Education'].unique():
    print(f'  {e}: response rate = {customers.loc[customers["Education"] == e, "Response"].mean():.3f}')

ANOVA, Response across education levels: F = 5.84, p = 1.139e-04
  Graduation: response rate = 0.136
  PhD: response rate = 0.210
  Master: response rate = 0.153
  Basic: response rate = 0.037
  2n Cycle: response rate = 0.110


In [4]:
contingency = pd.crosstab(customers['Kidhome'] > 0, customers['Response'])
chi2, p_kids, _, _ = stats.chi2_contingency(contingency)
print(f'has kids at home x Response: chi2 = {chi2:.2f}, p = {p_kids:.3e}')
contingency

has kids at home x Response: chi2 = 11.13, p = 8.512e-04


Response,0,1
Kidhome,,
False,1062,221
True,821,112


Income and total spend are strongly correlated - the highest-income customers spend meaningfully more. Education shows a real, if modest, association with campaign response (PhDs respond noticeably more often than customers with only a Basic education). Having kids at home has a large, significant negative association with response - unsurprising if these campaigns skew toward discretionary, non-essential goods.

In [5]:
import json
import os

os.makedirs('outputs', exist_ok = True)
results = {
    'income_spend_r': float(r_income_spend), 'income_spend_p': float(p_income_spend),
    'education_response_F': float(f_stat), 'education_response_p': float(p_edu),
    'kids_response_chi2': float(chi2), 'kids_response_p': float(p_kids),
    'response_rate_by_education': customers.groupby('Education')['Response'].mean().to_dict(),
}
with open('outputs/stats_results.json', 'w') as f:
    json.dump(results, f, indent = 2)

results

{'income_spend_r': 0.667576090388828,
 'income_spend_p': 5.843958806410119e-286,
 'education_response_F': 5.835583440494266,
 'education_response_p': 0.00011388415491246321,
 'kids_response_chi2': 11.126229666323527,
 'kids_response_p': 0.0008511542957802852,
 'response_rate_by_education': {'2n Cycle': 0.11,
  'Basic': 0.037037037037037035,
  'Graduation': 0.13620071684587814,
  'Master': 0.15342465753424658,
  'PhD': 0.20997920997921}}